In [33]:
import torch

CSV_PATH = "datasets/real_and_fake.csv"

BASE_DIR = "datasets"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [34]:
print(DEVICE)

cuda


In [35]:
import os

FRAME_DIR = "processed_frames"

REAL_FRAME_DIR = os.path.join(FRAME_DIR, "real")
ATTACK_FRAME_DIR = os.path.join(FRAME_DIR, "attack")

os.makedirs(REAL_FRAME_DIR, exist_ok=True)
os.makedirs(ATTACK_FRAME_DIR, exist_ok=True)

In [36]:
import pandas as pd
df = pd.read_csv(CSV_PATH, delimiter=",")
print(df.head())

                     file    type  split
0  train/real_video/0.mp4    real  train
1      train/attack/0.mp4  attack  train
2  train/real_video/1.mp4    real  train
3      train/attack/1.mp4  attack  train
4  train/real_video/2.mp4    real  train


In [37]:
import cv2

def extract_frames(video_path, output_dir, prefix, every_n_frame=10):
    cap = cv2.VideoCapture(video_path)

    frame_count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if frame_count % every_n_frame == 0:
            filename = f"{prefix}_{saved_count}.jpg"
            save_path = os.path.join(
                output_dir,
                filename
            )
            cv2.imwrite(save_path, frame)
            saved_count += 1
        frame_count += 1
    cap.release()

In [38]:
from tqdm import tqdm

print("[INFO] Extracting frames...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    video_rel_path = row["file"]
    label = row["type"]

    video_path = os.path.join(BASE_DIR, video_rel_path)
    if label == "real":
        output_dir = REAL_FRAME_DIR
    else:
        output_dir = ATTACK_FRAME_DIR

    extract_frames(
        video_path=video_path,
        output_dir=output_dir,
        prefix=f"{label}_{idx}"
    )

print("[INFO] Frame extraction complete.")

[INFO] Extracting frames...


100%|████████████████████████████████████████████████████████████████████████████████| 160/160 [05:12<00:00,  1.95s/it]

[INFO] Frame extraction complete.


In [39]:
import torch 
from PIL import Image
from torch.utils.data import Dataset

class LivenessDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

In [40]:
import os

image_paths = []
labels = []

# REAL
for img in os.listdir(REAL_FRAME_DIR):
    image_paths.append(os.path.join(REAL_FRAME_DIR, img))
    labels.append(1)

# ATTACK
for img in os.listdir(ATTACK_FRAME_DIR):
    image_paths.append(os.path.join(ATTACK_FRAME_DIR, img))
    labels.append(0)

print(f"[INFO] Total images: {len(image_paths)}")

[INFO] Total images: 5725


In [41]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    image_paths,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [42]:
import torchvision.transforms as transforms

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [43]:
from torch.utils.data import DataLoader

train_dataset = LivenessDataset(
    X_train,
    y_train,
    transform=train_transform
)

test_dataset = LivenessDataset(
    X_test,
    y_test,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [44]:
import torch.nn as nn

class CDCBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.conv(x)
        # central difference approximation
        center = out.mean(dim=(2,3), keepdim=True)

        out = out - center
        out = self.bn(out)
        out = self.relu(out)

        return out

In [45]:
class SimpleCDCN(nn.Module):

    def __init__(self):
        super().__init__()

        self.layer1 = CDCBlock(3, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.layer2 = CDCBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.layer3 = CDCBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Linear(128, 1)

    def forward(self, x):

        x = self.pool1(self.layer1(x))
        x = self.pool2(self.layer2(x))
        x = self.pool3(self.layer3(x))

        x = self.global_pool(x)

        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

In [46]:
model = SimpleCDCN().to(DEVICE)

In [47]:
import torch
import torch.nn as nn

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [48]:
print("[INFO] Start training...")

for epoch in range(EPOCHS):
    model.train()

    running_loss = 0

    correct = 0
    total = 0

    loop = tqdm(train_loader)

    for images, labels in loop:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images).squeeze()

        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        accuracy = 100 * correct / total

        loop.set_description(
            f"Epoch [{epoch+1}/{EPOCHS}]"
        )

        loop.set_postfix(
            loss=loss.item(),
            acc=accuracy
        )

    print(
        f"Epoch {epoch+1} "
        f"| Loss: {running_loss:.4f} "
        f"| Accuracy: {accuracy:.2f}%"
    )

[INFO] Start training...


Epoch [1/10]: 100%|██████████████████████████████████████████████| 144/144 [01:29<00:00,  1.62it/s, acc=69, loss=0.591]


Epoch 1 | Loss: 89.2527 | Accuracy: 69.00%


Epoch [2/10]: 100%|████████████████████████████████████████████| 144/144 [01:12<00:00,  1.98it/s, acc=74.3, loss=0.579]


Epoch 2 | Loss: 81.2502 | Accuracy: 74.32%


...

Epoch 8 | Loss: 61.4581 | Accuracy: 83.89%


Epoch [9/10]: 100%|████████████████████████████████████████████| 144/144 [01:26<00:00,  1.66it/s, acc=84.9, loss=0.335]


Epoch 9 | Loss: 59.0782 | Accuracy: 84.89%


Epoch [10/10]: 100%|███████████████████████████████████████████| 144/144 [01:25<00:00,  1.68it/s, acc=85.8, loss=0.231]

Epoch 10 | Loss: 57.1647 | Accuracy: 85.76%
